In [0]:
# ==============================================================================
#  DATA MARTS DE INSIGHTS DE NEGÓCIO: ECOMMERCE_CLIENTES
# ==============================================================================
TABELA_ALVO = "ecommerce_rastreamento"
print(f"\nGerando Insights de Negócio na memória para {TABELA_ALVO}...")

df_insight = None

# 1. Lê os logs de qualidade DIRETAMENTE do Data Lake para torná-lo independente
if delta_existe(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS):
    df_logs = ler_delta(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)
else:
    df_logs = None

# 2. Processa os insights
if df_logs is not None and df_logs.count() > 0:
    df_insight = df_logs \
        .withColumn("data_execucao", F.to_date("timestamp_execucao")) \
        .groupBy("data_execucao", "regra") \
        .agg(F.sum("qtd_registros_falhos").alias("volume_falhas")) \
        .orderBy(F.col("data_execucao").desc(), F.col("volume_falhas").desc())
else:
    print(f"-> Base limpa! Nenhum erro encontrado para {TABELA_ALVO}.")

# 3. Exibição na Tela
if df_insight is not None:
    print(f"-> Insight gerado com sucesso para {TABELA_ALVO}:")
    display(df_insight)
else:
    print("-> Nenhum insight a ser exibido nesta execução.")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# 3. Exibição na Tela e Gráfico
if df_insight is not None:
    print(f"-> Insight gerado com sucesso para {TABELA_ALVO}:")
    
    # 3.1. Mostra a tabela padrão (se quiser conferir os números exatos)
    display(df_insight)
    
    # 3.2. Converte para Pandas (Seguro, pois é apenas uma tabela agregada pequena)
    df_pd = df_insight.toPandas()
    
    # 3.3. Configura e gera o gráfico
    plt.figure(figsize=(10, 6))
    
    # Exemplo genérico que funciona bem para quase todas as suas tabelas:
    # Eixo X = A primeira coluna do groupBy (Data, Categoria ou Regra)
    # Eixo Y = A coluna que tem a métrica calculada
    coluna_x = df_pd.columns[1] # Pega a coluna de regra/categoria/status
    coluna_y = df_pd.columns[-1] # Pega a última coluna que normalmente é a de volume/faturamento
    
    sns.barplot(data=df_pd, x=coluna_x, y=coluna_y, palette="viridis")
    
    plt.title(f"Análise de Qualidade/Negócio - {TABELA_ALVO}", fontsize=14, fontweight='bold')
    plt.xlabel(coluna_x.replace("_", " ").title(), fontsize=12)
    plt.ylabel(coluna_y.replace("_", " ").title(), fontsize=12)
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("-> Nenhum insight a ser exibido nesta execução.")